In [ ]:
import os
import time
import json
import requests
from urllib.parse import quote_plus

# Tree-sitter core and language bindings
import tree_sitter_python as tspython
import tree_sitter_java as tsjava
import tree_sitter_javascript as tsjs
import tree_sitter_typescript as tsts
import tree_sitter_c as tsc
import tree_sitter_c_sharp as tscsharp
import tree_sitter_cpp as tscpp
from tree_sitter import Language, Parser

In [ ]:
ALGORITHM_CLASSES = {
    "monolithic_hash_functions": [
        "md5", "sha1", "sha256", "sha512", "sha3", "hmac", "blake2s", "blake2b"
    ],
    "iterative_hash_functions": [
        "pbkdf", "pbkdf2", "bcrypt", "crypt", "scrypt"
    ],
    "memory_hard_functions": [
        "yescrypt", "argon2id", "argon2i", "argon2d", "balloon hashing", "lyra2"
    ],
}

CONTEXT_KEYWORDS = ["password", "hash", "login", "auth"]
LANGUAGES = ["c"] #Change LANGUAGES to include the languages you want to analyse on each run
TOKEN = "temporary_token"  # Replace with your actual GitHub Personal Access Token

In [ ]:
import json
from tree_sitter import Language, Parser, Query, QueryCursor

class CodeScanner:
    def __init__(self, config_path="functions_config.json"):
        with open(config_path, 'r') as f:
            self.config = json.load(f)
            
        self.parsers = {}
        self.queries = {}
        self._setup_languages()

    def _setup_languages(self):
        # Precise AST query strings per language specification
        languages_meta = {
            'python': (tspython.language(), '(call function: (_) @func arguments: (argument_list) @args)'),
            'java': (tsjava.language(), '(method_invocation name: (_) @func arguments: (argument_list) @args)'),
            'javascript': (tsjs.language(), '(call_expression function: (_) @func arguments: (arguments) @args)'),
            'typescript': (tsts.language_typescript(), '(call_expression function: (_) @func arguments: (arguments) @args)'),     
            'c': (tsc.language(), '(call_expression function: (_) @func arguments: (argument_list) @args)'),
            'csharp': (tscsharp.language(), '(invocation_expression expression: (_) @func (argument_list) @args)'),
            'cpp': (tscpp.language(), '(call_expression function: (_) @func arguments: (argument_list) @args)')
        }

        for lang_key, (lang_ptr, query_str) in languages_meta.items():
            try:
                lang_obj = Language(lang_ptr)
                parser = Parser(lang_obj)
                query = Query(lang_obj, query_str)
                
                self.parsers[lang_key] = parser
                self.queries[lang_key] = query
            except Exception as e:
                print(f"Failed to load parser for {lang_key}: {e}")

    def scan_file_content(self, file_content: str, language_key: str):
        if language_key not in self.parsers or language_key not in self.config:
            return []

        parser = self.parsers[language_key]
        query = self.queries[language_key]
        
        tree = parser.parse(bytes(file_content, "utf8"))
        
        # Tree-sitter 0.22+ uses QueryCursor to execute AST queries
        cursor = QueryCursor(query)
        matches = cursor.matches(tree.root_node)
        findings = []

        for match in matches:
            func_node = None
            args_node = None

            # Unpack matches across tuple/list structures
            captures = match[1] if isinstance(match, (tuple, list)) else getattr(match, "captures", match)
            
            if isinstance(captures, dict):
                func_nodes = captures.get("func", [])
                args_nodes = captures.get("args", [])
                func_node = func_nodes[0] if func_nodes else None
                args_node = args_nodes[0] if args_nodes else None
            else:
                for node, capture_name in captures:
                    if capture_name == "func":
                        func_node = node
                    elif capture_name == "args":
                        args_node = node

            if func_node:
                func_name = func_node.text.decode("utf8")
                raw_args = args_node.text.decode("utf8") if args_node else "()"
                
                config_for_lang = self.config[language_key]
                matched_key = None

                # 1. Direct match on function call (e.g. BCrypt.hashpw or MessageDigest.getInstance)
                if func_name in config_for_lang:
                    matched_key = func_name
                else:
                    # 2. Argument match for Java algorithms (e.g. SHA-256, MD5 passed inside getInstance("SHA-256"))
                    for key in config_for_lang.keys():
                        if key.lower() in raw_args.lower():
                            matched_key = key
                            break

                # If either condition matched, record the finding
                if matched_key:
                    findings.append({
                        "function": func_name,
                        "parameters": raw_args,
                        "line": func_node.start_point[0] + 1,
                        "algorithm_data": config_for_lang[matched_key]
                    })

        return findings

In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

# Only auto-retry on server-side errors (500s), NOT rate limits (403/429)
session = requests.Session()
retries = Retry(total=3, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

def build_search_url(query: str, language: str, context: str = "", page: int = 1) -> str:
    clean_query = f'"{query}"' if " " in query else query
    full_query = f"{clean_query} {context} language:{language}" if context else f"{clean_query} language:{language}"
    encoded_query = quote_plus(full_query)
    return f"https://api.github.com/search/code?q={encoded_query}&per_page=100&page={page}"

def make_request(url: str, token: str | None = None, raw_mode: bool = False):
    headers = {
        "User-Agent": "CyberSecurity-Master-Student-Project",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    
    if raw_mode:
        headers["Accept"] = "application/vnd.github.raw+json"
    else:
        headers["Accept"] = "application/vnd.github+json"
        
    if token and token != "inserttokenhere":
        headers["Authorization"] = f"Bearer {token}"

    while True:
        try:
            # Pinging Search API vs Raw File CDN
            if not raw_mode:
                time.sleep(2.0)  # Keep search requests under 30 req/min secondary rate limit
                
            response = session.get(url, headers=headers, timeout=30)
            
            # Handle rate limits manually
            if response.status_code in (403, 429):
                reset_header = response.headers.get("X-RateLimit-Reset")
                retry_after_header = response.headers.get("Retry-After")
                
                if reset_header:
                    sleep_time = max(int(reset_header) - int(time.time()), 5)
                elif retry_after_header:
                    sleep_time = int(retry_after_header)
                else:
                    sleep_time = 60
                    
                print(f"\n GitHub Rate Limit hit. Waiting {sleep_time}s for reset...")
                time.sleep(sleep_time + 1)
                continue  # Retry request after waiting out rate limit
                
            return response
            
        except Exception as e:
            print(f"\n[!] Network hiccup: {e}. Retrying in 5s...")
            time.sleep(5)

In [ ]:
import concurrent.futures

# Keep track of URLs already processed to avoid duplicates across contexts
seen_urls = set()

def fetch_and_parse_worker(item, language, scanner):
    """ Function to download and parse a single file."""
    api_url = item.get("url")
    html_url = item.get("html_url")
    
    # Deduplication check
    if html_url in seen_urls:
        return None
        
    code_response = make_request(api_url, TOKEN, raw_mode=True)
    if code_response and code_response.status_code == 200:
        seen_urls.add(html_url)
        raw_code = code_response.text
        findings = scanner.scan_file_content(raw_code, language)
        if findings:
            return {"item": item, "findings": findings}
    return None

def process_and_save_results(
    response, language: str, algorithm: str, algorithm_class: str, scanner, context: str = "", page: int = 1
) -> int:
    folder_algo_name = algorithm.replace(" ", "_").lower()
    output_dir = f"../master_dataset/{language}/{folder_algo_name}"
    os.makedirs(output_dir, exist_ok=True)
    filepath = f"{output_dir}/{folder_algo_name}_matches.txt"

    if not response or response.status_code != 200:
        return 0

    items = response.json().get("items", [])
    valid_findings = []

    # MULTITHREADING: Process up to 15 files concurrently
    with concurrent.futures.ThreadPoolExecutor(max_workers=15) as executor:
        futures = [executor.submit(fetch_and_parse_worker, item, language, scanner) for item in items]
        
        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            if result:
                valid_findings.append(result)

    # SEQUENTIAL WRITE: Safely write all gathered findings to the text file
    mode = "w" if page == 1 and context == CONTEXT_KEYWORDS[0] else "a"
    with open(filepath, mode, encoding="utf-8") as f:
        if page == 1 and context == CONTEXT_KEYWORDS[0]:
            f.write(f"=== AST Extracted Parameters ===\n")
            f.write(f"Class: {algorithm_class.upper()} | Algo: {algorithm.upper()} | Lang: {language.upper()}\n")
            f.write("=" * 60 + "\n\n")

        for entry in valid_findings:
            item = entry["item"]
            repo = item.get("repository", {}).get("full_name")
            f.write(f"Repository: {repo}\n")
            f.write(f"    File: {item.get('path')}\n")
            f.write(f"    URL: {item.get('html_url')}\n")
            for find in entry["findings"]:
                f.write(f"    -> Line {find['line']}: {find['function']}\n")
                f.write(f"    -> Parameters: {find['parameters']}\n")
            f.write("-" * 40 + "\n")

    # Clear print output showing the active context keyword!
    print(f"Scanned page {page} ({len(items)} files, {len(valid_findings)} matches) for {algorithm.upper()} in {language.upper()} [Context: '{context}']")
    return len(items)

In [ ]:
import time
from datetime import datetime

print("Initialising Tree-sitter parsers...")
scanner = CodeScanner("functions_config.json")
print("Parsers loaded successfully.\n")

if TOKEN == "inserttokenhere":
    print("Warning: No token found. You will be severely rate limited.")

for algorithm_class, algos in ALGORITHM_CLASSES.items():
    print(f"\n Processing Classification: {algorithm_class.upper()}")
    print("=" * 50)
    
    for algo in algos:
        for lang in LANGUAGES:
            folder_algo_name = algo.replace(" ", "_").lower()
            filepath = f"../master_dataset/{lang}/{folder_algo_name}/{folder_algo_name}_matches.txt"
            
            # RESUME FEATURE: Skip if file already exists and isn't empty
            if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
                print(f" Skipping {algo.upper()} in {lang.upper()} (Already downloaded)")
                continue

            seen_urls.clear()                         

            for context in CONTEXT_KEYWORDS:
                page = 1
                while True:
                    url = build_search_url(algo, lang, context=context, page=page)
                    try:
                        search_response = make_request(url, TOKEN)
                        
                        # --- RATE LIMIT CHECK & COUNTDOWN ---
                        if search_response.status_code in (403, 429):
                            reset_timestamp = int(search_response.headers.get("X-RateLimit-Reset", time.time() + 60))
                            sleep_seconds = max(int(reset_timestamp - time.time()) + 2, 5)
                            reset_time_str = datetime.fromtimestamp(reset_timestamp).strftime('%H:%M:%S')

                            print(f"\n[ RATE LIMIT HIT] Sleeping for {sleep_seconds}s. Resuming at {reset_time_str}...")

                            for remaining in range(sleep_seconds, 0, -5):
                                print(f"    Resuming in {remaining}s...", end="\r")
                                time.sleep(min(5, remaining))

                            print(f"\n[ Resuming Collection] Rate limit reset at {datetime.now().strftime('%H:%M:%S')}. Retrying request...\n")
                            continue  # Retry the exact same page request without incrementing page
                        # --- END RATE LIMIT CHECK ---

                        # Pass context=context so progress prints clearly
                        items_processed = process_and_save_results(
                            search_response, lang, algo, algorithm_class, scanner, context=context, page=page
                        )
                        
                        # Stop if  hit the end of search results or GitHub's 10-page limit
                        if items_processed < 100 or page >= 10:
                            break
                            
                        page += 1

                    except Exception as exc:
                        print(f" Error processing {algo} in {lang} (Context: '{context}', Page {page}): {exc}")
                        break